<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🗺️ Step 1 — Spatial Join with Neighborhoods</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Map each property to its Vancouver neighborhood using GeoJSON</p>
</div>

In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

print("✅ Libraries imported!")

✅ Libraries imported!


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">📊 Step 2 — Load Housing Data</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Load cleaned Vancouver properties with coordinates</p>
</div>

In [4]:
# Load the cleaned housing data from previous notebook
df = pd.read_csv('../data/processed/vancouver_housing_cleaned.csv')

print(f"✅ Housing data loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

✅ Housing data loaded: (3375, 16)
Columns: ['Price', 'Bedrooms', 'Bathrooms', 'Square Footage', 'Latitude', 'Longitude', 'Acreage', 'Garage', 'Parking', 'Fireplace', 'Heating', 'Waterfront', 'Sewer', 'Pool', 'Garden', 'Balcony']

First few rows:
       Price  Bedrooms  Bathrooms  Square Footage   Latitude   Longitude  \
0  1798000.0       2.0        3.0          1183.0  49.286844 -123.124152   
1   928000.0       2.0        2.0           884.0  49.215653 -123.116343   
2   918000.0       2.0        2.0           815.0  49.215653 -123.116343   
3   729000.0       1.0        1.0           708.0  49.277631 -123.122958   
4   899000.0       1.0        1.0           862.0  49.275778 -123.125761   

   Acreage Garage Parking Fireplace     Heating Waterfront Sewer Pool Garden  \
0      0.0    Yes     Yes        No  forced air         No  none   No     No   
1      0.0    Yes     Yes        No   heat pump         No  none   No     No   
2      0.0    Yes     Yes        No   heat pump         N

<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🗺️ Step 3 — Load Neighborhood Boundaries</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Load Vancouver's 22 neighborhood GeoJSON boundaries</p>
</div>

In [5]:
# Load Vancouver neighborhood boundaries
gdf_boundaries = gpd.read_file('../data/raw/local-area-boundary.geojson')

print(f"✅ Neighborhood boundaries loaded: {gdf_boundaries.shape}")
print(f"\nNeighborhoods:")
print(gdf_boundaries['name'].tolist())
print(f"\nCRS: {gdf_boundaries.crs}")

✅ Neighborhood boundaries loaded: (22, 3)

Neighborhoods:
['Arbutus Ridge', 'Grandview-Woodland', 'Killarney', 'Strathcona', 'Sunset', 'Hastings-Sunrise', 'Kerrisdale', 'South Cambie', 'Riley Park', 'Shaughnessy', 'Victoria-Fraserview', 'West Point Grey', 'Mount Pleasant', 'Renfrew-Collingwood', 'West End', 'Downtown', 'Marpole', 'Oakridge', 'Dunbar-Southlands', 'Fairview', 'Kensington-Cedar Cottage', 'Kitsilano']

CRS: EPSG:4326


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🔗 Step 4 — Perform Spatial Join</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Map each property to its neighborhood using latitude/longitude</p>
</div>

In [6]:
# Convert housing data to GeoDataFrame using latitude/longitude
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf_housing = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

print(f"✅ Housing GeoDataFrame created: {gdf_housing.shape}")

# Ensure both have same coordinate system
gdf_boundaries = gdf_boundaries.to_crs('EPSG:4326')

# Spatial join - each property gets assigned to its neighborhood
joined = gpd.sjoin(gdf_housing, gdf_boundaries[['name', 'geometry']], 
                   how='left', predicate='within')

print(f"✅ Spatial join complete!")
print(f"Shape: {joined.shape}")
print(f"\nNeighborhoods found:")
print(joined['name'].value_counts())

✅ Housing GeoDataFrame created: (3375, 17)
✅ Spatial join complete!
Shape: (3375, 19)

Neighborhoods found:
name
Downtown                    790
Marpole                     219
Renfrew-Collingwood         215
Mount Pleasant              181
West End                    175
Kensington-Cedar Cottage    140
Dunbar-Southlands           131
Hastings-Sunrise            122
Oakridge                    116
Kitsilano                   109
Kerrisdale                  109
Killarney                   108
Fairview                    105
Sunset                      102
Riley Park                  102
Grandview-Woodland           97
South Cambie                 95
West Point Grey              87
Shaughnessy                  74
Arbutus Ridge                74
Victoria-Fraserview          64
Strathcona                   34
Name: count, dtype: int64


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">📈 Step 5 — Analyze Neighborhoods</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Compare prices and features across Vancouver's 22 neighborhoods</p>
</div>

In [7]:
# Clean up the data
df_neighborhoods = joined.drop(columns=['geometry', 'index_right']).copy()

print(f"✅ Cleaned data: {df_neighborhoods.shape}")

# Calculate neighborhood statistics
neighborhood_stats = df_neighborhoods.groupby('name').agg({
    'Price': ['count', 'mean', 'median', 'min', 'max', 'std'],
    'Bedrooms': 'mean',
    'Bathrooms': 'mean',
    'Square Footage': 'mean',
    'Latitude': 'mean',
    'Longitude': 'mean'
}).round(2)

# Flatten column names
neighborhood_stats.columns = ['_'.join(col).strip() for col in neighborhood_stats.columns.values]
neighborhood_stats = neighborhood_stats.sort_values('Price_mean', ascending=False)

print("\n💰 NEIGHBORHOOD PRICE ANALYSIS (Sorted by Average Price):")
print(neighborhood_stats)

✅ Cleaned data: (3375, 17)

💰 NEIGHBORHOOD PRICE ANALYSIS (Sorted by Average Price):
                          Price_count  Price_mean  Price_median  Price_min  \
name                                                                         
Shaughnessy                        74  8171901.35     7185000.0   978000.0   
West Point Grey                    87  6109759.84     4050000.0   599900.0   
Kerrisdale                        109  5712916.50     4180000.0   479000.0   
Dunbar-Southlands                 131  3951581.15     3788000.0   899000.0   
Arbutus Ridge                      74  3512995.74     3514000.0   479000.0   
Oakridge                          116  2980946.45     1893000.0   499000.0   
South Cambie                       95  2524667.36     1868000.0   644000.0   
Kitsilano                         109  2140022.68     1748800.0   529800.0   
Hastings-Sunrise                  122  2020690.01     1748900.0   509000.0   
Riley Park                        102  1994004.65     189

<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">💾 Step 6 — Save Neighborhood Data</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Save neighborhood analysis and full dataset with neighborhood assignments</p>
</div>

In [8]:
# Save neighborhood statistics
neighborhood_stats.to_csv('../data/processed/neighborhood_analysis.csv')

# Save full dataset with neighborhoods
df_neighborhoods.to_csv('../data/processed/vancouver_with_neighborhoods.csv', index=False)

print("✅ Neighborhood analysis saved!")
print("✅ Full dataset with neighborhoods saved!")
print("\nFiles created:")
print("  1. neighborhood_analysis.csv")
print("  2. vancouver_with_neighborhoods.csv")

✅ Neighborhood analysis saved!
✅ Full dataset with neighborhoods saved!

Files created:
  1. neighborhood_analysis.csv
  2. vancouver_with_neighborhoods.csv


<div style="background-color:#1a1a2e; padding:20px; border-radius:10px;">

<h3 style="color:#00d4ff;">✅ What We Accomplished</h3>
<ul style="color:#a8a8b3; line-height: 2;">
<li>✓ Mapped 3,375 Vancouver properties to their neighborhoods using GeoJSON</li>
<li>✓ Identified all 22 Vancouver neighborhoods in our dataset</li>
<li>✓ Calculated neighborhood-level price statistics</li>
<li>✓ Found 8x price difference: Shaughnessy ($8.17M) vs Strathcona ($1.03M)</li>
<li>✓ Discovered expensive neighborhoods have larger homes (more beds, baths, sq ft)</li>
</ul>

<h3 style="color:#00d4ff; margin-top:20px;">🏙️ Top 3 Most Expensive Neighborhoods</h3>
<table style="color:#a8a8b3; width:100%; margin-top:10px;">
<tr style="border-bottom: 1px solid #444;">
<td><b>Shaughnessy</b></td>
<td>$8.17M avg</td>
<td>5.5 beds</td>
<td>5,635 sq ft</td>
</tr>
<tr style="border-bottom: 1px solid #444;">
<td><b>West Point Grey</b></td>
<td>$6.11M avg</td>
<td>4.2 beds</td>
<td>3,133 sq ft</td>
</tr>
<tr style="border-bottom: 1px solid #444;">
<td><b>Kerrisdale</b></td>
<td>$5.71M avg</td>
<td>4.39 beds</td>
<td>3,775 sq ft</td>
</tr>
</table>

<h3 style="color:#00d4ff; margin-top:20px;">💡 Key Insights</h3>
<ul style="color:#a8a8b3; line-height: 2;">
<li>📍 <b>Location matters:</b> Neighborhood explains massive price variation</li>
<li>🏠 <b>Size correlates:</b> Expensive neighborhoods have larger homes</li>
<li>🌊 <b>West side premium:</b> West Vancouver neighborhoods command 5-8x prices</li>
<li>🏢 <b>Downtown different:</b> Most properties but lower average (condos)</li>
<li>🎯 <b>Next step:</b> Use Linear Regression & Random Forest to quantify feature importance</li>
</ul>

<h3 style="color:#00d4ff; margin-top:20px;">📁 Files Created</h3>
<ul style="color:#a8a8b3;">
<li>neighborhood_analysis.csv — Summary statistics by neighborhood</li>
<li>vancouver_with_neighborhoods.csv — Full dataset with neighborhood names</li>
</ul>

</div>